# AgentIR · Multiple-query and history-preserving retrieval pilot

**Run on Google Colab with a GPU.** Select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU).

This notebook runs **Qwen/Qwen3-4B-Instruct-2507 locally on the GPU**, using 4-bit NF4 weights. **No Qwen API key, Hugging Face token, or paid inference endpoint is required.** The public weights download automatically. Colab GPU availability depends on your account.

We compare four methods on the same saved PlugIR descriptions and 50,000 image embeddings.

| Method | Search descriptions combined |
| --- | --- |
| A | Current description |
| B | Current description + two generated variants |
| C | Original description + current description |
| D | Original description + current description + variant 1 |

B versus A tests multiple query perspectives. C versus A tests retaining the original description. D versus B compares three searches on each side to test the value of earlier information.

The notebook first checks 10 targets, then runs a **separate 100-target pilot**. Both use saved states after 4 and 9 answers. The samples exclude the three discussed examples and indices from our earlier 50-target Qwen pilot. All targets and settings are chosen before viewing these results.

**Files to keep together**
- This notebook, `AgentIR_Multi_Query_Pilot.ipynb`
- `multi_query_pilot_bundle.zip` (about 48 MB)

Upload the notebook through **File → Upload notebook** in Colab. The data cell below asks for the ZIP once and stores it in your Drive. Share both files with teammates. Each teammate should use their own Drive output folder.


## 1. Install the inference packages
Run this cell first. It keeps Colab's installed PyTorch and CUDA pair and uses the inference-package versions recorded in our successful T4 configuration. If Colab asks for a runtime restart, choose **Restart session**, then rerun this cell and continue.

In [ ]:
%pip -q install "transformers==4.51.3" "accelerate==1.15.0" "bitsandbytes==0.48.2" "huggingface-hub==0.36.2" "pandas>=2,<4"


## 2. Check the GPU and connect Drive
Approve the normal Drive connection prompt. Results and query-generation checkpoints go to your Drive, allowing a disconnected session to resume. Model downloads use Colab's local disk.

Choose a new `RUN_NAME` if you deliberately change the experiment code or prompts. Reuse the same name to resume an interrupted run. Avoid running two sessions into the same output folder simultaneously.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/agentir_model_cache"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/AgentIR_multi_query")
RUN_NAME = "pilot_v1"
RUN_DIR = PROJECT_DIR / "runs" / RUN_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved to:", RUN_DIR)


## 3. Load the provided data bundle
On the first run, select **multi_query_pilot_bundle.zip** from your computer. Subsequent runs reuse the saved copy in Drive. The ZIP includes the cached image vectors, corpus IDs, saved descriptions, and experiment code. It contains no model weights or API credentials.

The checksum checks that everyone is using exactly the same bundle.

In [ ]:
import hashlib
import zipfile
from google.colab import files

EXPECTED_SHA256 = "4418ddc07374ab4b42d1c0b97f034afc3580e9adf28d76e6f81da2087554b9db"
BUNDLE_PATH = PROJECT_DIR / "multi_query_pilot_bundle.zip"
if not BUNDLE_PATH.exists():
    uploaded = files.upload()
    matches = [value for name, value in uploaded.items() if name == "multi_query_pilot_bundle.zip"]
    if len(matches) != 1:
        raise ValueError("Upload the file named multi_query_pilot_bundle.zip.")
    if hashlib.sha256(matches[0]).hexdigest() != EXPECTED_SHA256:
        raise ValueError("The uploaded ZIP does not match this notebook version.")
    BUNDLE_PATH.write_bytes(matches[0])
    del uploaded, matches

if hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError("The ZIP in Drive differs from this notebook. Use the matching notebook and bundle.")

WORK_DIR = Path("/content/agentir_multi_query")
WORK_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for name in archive.namelist():
        if Path(name).name != name:
            raise ValueError("Unexpected path in the bundle.")
    archive.extractall(WORK_DIR)

import json
manifest = json.loads((WORK_DIR / "bundle_manifest.json").read_text())
print("Smoke targets:", len(manifest["splits"]["smoke"]))
print("Pilot targets:", len(manifest["splits"]["pilot"]))
print("Image candidates: 50,000")
print("Qwen revision: cdbee75f17c01a7cc42f958dc650907174af0554")
print("BLIP revision: 19502f1e215844f7e48bd48473f86932486d3441")


## 4. Run the 10-target setup check
This generates two complementary variants for each current description. The model receives only that description. It does not receive target photos, image IDs, retrieval scores, earlier descriptions, or future states.

Qwen generation and BLIP retrieval run in separate processes, releasing Qwen's GPU memory before BLIP loads. Completed generations are reused on reruns. Invalid JSON or duplicate variants receive one retry, and persistent failures stop evaluation for inspection.

The first run downloads roughly 8 GB of Qwen source weights plus the BLIP checkpoint; Qwen is quantized when loaded. Download time depends on the connection. The generation log shows progress state by state.

In [ ]:
import subprocess
import sys
import time

def run_phase(split):
    output = RUN_DIR / split
    phase_start = time.perf_counter()
    for stage in ("generate", "evaluate"):
        print(f"{split.upper()} / {stage.upper()}", flush=True)
        subprocess.run(
            [sys.executable, str(WORK_DIR / "experiment.py"), stage,
             "--output", str(output), "--split", split],
            cwd=WORK_DIR, check=True,
        )
    elapsed = time.perf_counter() - phase_start
    print(f"This invocation completed in {elapsed / 60:.1f} minutes.")
    print("Saved results:", output)
    return output

SMOKE_DIR = run_phase("smoke")


## 5. Inspect the setup check
Check a few generated descriptions for invented or omitted facts. The automated checks validate formatting and distinctness; they cannot prove that every description is faithful.

If prompts or settings need changes, make them using this smoke sample, then use a new run folder. Keep the 100-target pilot fixed and report its first run.

The paired comparisons show both improvements and regressions. `D vs B` is the equal-search-count comparison.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

display(pd.read_csv(SMOKE_DIR / "summary.csv").round(4))
display(pd.read_csv(SMOKE_DIR / "comparisons.csv").round(2))
pd.set_option("display.max_colwidth", 180)
display(pd.read_csv(SMOKE_DIR / "queries_for_review.csv").head(6))
timing = json.loads((SMOKE_DIR / "timing.json").read_text())
print("Measured request time per saved state (seconds):",
      round(timing["generation_request_seconds"] / 20, 2))
print("Rough Qwen request-time projection for 200 pilot states (minutes):",
      round(timing["generation_request_seconds"] * 10 / 60, 1))
print("This projection excludes downloads and model loading.")
print("Truncated search descriptions:", timing["truncated_text_count"])


## 6. Run the 100-target pilot
Run after checking the smoke outputs above. This evaluates **200 saved states** using 100 separate targets and the same four methods.

If Colab disconnects, reopen this notebook and rerun the cells with the same `RUN_NAME`. Completed query generations are preserved in Drive. Retrieval evaluation can be rerun from those saved queries.

An interrupted cell will not count as a completed experiment. Persistent generation failures are written to `generation_failures.json`; inspect those alongside `llm_requests.jsonl` before retrying.

In [ ]:
PILOT_DIR = run_phase("pilot")


## 7. View the numerical comparison
**Recall@1 / @10 / @50** is the percentage of targets within that rank at the specified saved stage. **MRR** is the mean of 1 / target rank; higher is better.

The paired table reports the number of targets with improved, unchanged, or worsened ranks, plus top-10 recoveries and losses. `delta_recall10_pp` is the percentage-point difference. Its 95% paired bootstrap interval resamples targets within each stage. The two stages are reported separately.

Searches share one frozen retriever, corpus, and scoring procedure. Fusion uses complete rankings and the fixed rule RRF with k=60. Exact duplicate searches contribute one vote. No targets are dropped because a method performs badly.

These numbers measure retrieval at saved PlugIR conversation states. They do not establish full interactive-system gains or improvements over a reproduced A2R baseline.

In [ ]:
display(Markdown((PILOT_DIR / "results.md").read_text()))
display(pd.read_csv(PILOT_DIR / "comparisons.csv").round(2))

summary = pd.read_csv(PILOT_DIR / "summary.csv")
axes = summary.pivot(index="method", columns="answers", values="recall_at_10").plot.bar(
    figsize=(8, 4), ylim=(0, 100), rot=0,
    ylabel="Targets in top 10 (%)", xlabel="Method",
    title="Paired retrieval pilot — Recall@10",
)
axes.legend(title="Answers already gathered")
axes.figure.tight_layout()
axes.figure.savefig(PILOT_DIR / "recall_at_10.png", dpi=160)
print("All output files:", PILOT_DIR)


## 8. Inspect gains and regressions, then download results
This lists examples in both directions. Use the full tables when presenting the results.

The ZIP includes the exact sample, model revisions, prompts and raw generation logs, generated descriptions, per-target ranks, top-10 image IDs, metrics, and timing. Share the results ZIP with teammates or upload it to this chat.

In [ ]:
ranks = pd.read_csv(PILOT_DIR / "ranks.csv")
paired = ranks.pivot(index=["source_index", "answers", "target_path"],
                     columns="method", values="target_rank").reset_index()
paired["B_rank_improvement"] = paired["A"] - paired["B"]
print("Largest B improvements")
display(paired.sort_values("B_rank_improvement", ascending=False).head(5))
print("Largest B regressions")
display(paired.sort_values("B_rank_improvement", ascending=True).head(5))
paired.to_csv(PILOT_DIR / "paired_ranks.csv", index=False)

import shutil
download_path = shutil.make_archive("/content/AgentIR_multi_query_results", "zip", RUN_DIR)
files.download(download_path)


### Reproducibility and interpretation
- Qwen is **Qwen/Qwen3-4B-Instruct-2507**, revision `cdbee75f17c01a7cc42f958dc650907174af0554`, with NF4 double quantization and float16 compute.
- Greedy decoding, fixed prompt, fixed samples, and cached outputs keep the comparison consistent. Different GPU types or inference libraries can still produce differences.
- BLIP uses **Salesforce/blip-itm-large-coco**, revision `19502f1e215844f7e48bd48473f86932486d3441`, matching the original project's frozen-BLIP cache provenance.
- Generation follows the supplied current description even if that description already contains an error. Review is therefore important.
- Generation-request time and shared retrieval time are recorded separately. Because searches are cached across methods, these timings describe the experiment, not independent production latency for each method.
- The data bundle was prepared from the existing project cache and saved PlugIR queries. This notebook does not recreate the authors' complete interactive system.

[Qwen model](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) · [BLIP model](https://huggingface.co/Salesforce/blip-itm-large-coco) · [PlugIR](https://aclanthology.org/2024.acl-long.46/) · [Reciprocal rank fusion](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf)
